# 09 Hyper-parameter search for Deep Q Learning

#### 👉Deep RL is hard, because (among other things) it's very sensitivity to the hyper-parameters.

#### 👉We tune the hyper-parmeters following a trial&error approach:

![](../images/hparams_search_diagram.svg)

#### 👉However, Hyper-parameter spaces in deep RL problems are HUGE. A brute-force solution that would try all possible combinations of hyper-parameters is not feasible. We need something smarter than that...

#### 👉And this is when Bayesian search methods enther into the picture.

#### 👉In a nutshell, Bayesian search methods use past searches to inform promising avenues.

#### 👉 [Optuna](https://optuna.readthedocs.io/en/stable/index.html) is a Python open-source library that implements Bayesian search methods

<img src="https://github.com/Paulescu/hands-on-rl/blob/main/03_cart_pole/images/optuna.png?raw=True" width="400"/>

#### 👉Hyper-paramater search a piece of cake 🍰if you use Optuna.

In [10]:
%load_ext autoreload
%autoreload 2
%pylab inline
%config InlineBackend.figure_format = 'svg'

%pylab is deprecated, use %matplotlib inline and import the required libraries.
Populating the interactive namespace from numpy and matplotlib


c:\Users\Ehtisham\anaconda3\envs\rlcodenv\lib\site-packages\IPython\core\magics\pylab.py:166: UserWarning: pylab import has clobbered these variables: ['random']
`%matplotlib` prevents importing * from pylab and numpy
  warn("pylab import has clobbered these variables: %s"  % clobbered +


In [1]:
import os
import gym
import numpy as np
import pandas as pd
import random
import time
from matplotlib import pyplot as plt
from openpyxl import load_workbook

In [2]:
# Run only if Kernel is dying due to matplotlib plt command
os.environ["KMP_DUPLICATE_LIB_OK"]="TRUE"

## Environment 🌎

In [3]:
# import gym
# env = gym.make('CartPole-v1')

from src.mg_env import *
env = MicroGridEnv()

### MLflow is a useful tool to track experiment results

cd to the root directory of this lesson (in my case `/Users/paulabartabajo/src/online-courses/hands-on-rl/03_cart_pole`) and spin up the mlflow tracking server as follows:

**$ mlflow server --backend-store-uri sqlite:///mlflow.db --default-artifact-root ./artifacts --host 0.0.0.0 --port 5000**

### 💡 if you have another service listening to port 5000, increase port number by 1 until you hit a free port.

In [14]:
%pwd

'c:\\RL-Git-Projects\\BMS_Hyperpar_search'

In [65]:
import mlflow

# connect mlflow client to the mlflow server that runs on localhost:5000
MLFLOW_SERVER_URI = 'http://localhost:5000'
mlflow.set_tracking_uri(str(MLFLOW_SERVER_URI))

EXPERIMENT_NAME = 'hyperparameter_search'
mlflow.set_experiment(EXPERIMENT_NAME)

<Experiment: artifact_location='file:///C:/RL-Git-Projects/BMS_Hyperpar_search/artifacts/1', creation_time=1730803118762, experiment_id='1', last_update_time=1730803118762, lifecycle_stage='active', name='hyperparameter_search', tags={}>

In [17]:
# with mlflow.start_run():
#     mlflow.log_param("param1", 5)

2024/11/05 10:51:04 INFO mlflow.tracking._tracking_service.client: 🏃 View run masked-wasp-112 at: http://localhost:5000/#/experiments/1/runs/118894f640f045e7be8edff6f3050d36.
2024/11/05 10:51:04 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/1.


## Create an Optuna study

In [66]:
import optuna

from src.config import OPTUNA_DB

study = optuna.create_study(
    study_name=EXPERIMENT_NAME,
    direction='maximize',
    load_if_exists=True,
    storage=f'sqlite:///{OPTUNA_DB}'
)

[I 2024-11-05 16:24:26,928] Using an existing study with name 'hyperparameter_search' instead of creating a new one.


## Objective function we want to maximize

In [67]:
from src.optimize_hyperparameters import objective

# we define a lambda function because study.optimize()
# expect the objective function to have only 1 input
# (trial), while our objective function hast 2 extra
# inputs I defined to add flexibility to the script
func = lambda trial: objective(trial,
                               force_linear_model=False,
                               n_episodes_to_train=2000)

## Set threshold to terminate hyperparameter search

In [68]:
class CheckHyperparamMeanRewardThreshold:
    def __init__(self, reward_threshold: float):
        self.reward_threshold = reward_threshold

    def __call__(self, study: optuna.study.Study, trial: optuna.trial.FrozenTrial) -> None:
        if trial.value >= self.reward_threshold:
            print((f'Stopping hyperparameter search because trial.value ({trial.value}) '
                   f'hit threshold ({self.reward_threshold})'))
            study.stop()

# Stop hyperparameter search when we hit a perfect mean reward of 500
hyperparam_search_stop_callback = CheckHyperparamMeanRewardThreshold(9364.0)

## Let's start the search

In [69]:
study.optimize(func, n_trials=200, callbacks=[hyperparam_search_stop_callback])

GPU is running
75,017 parameters


100%|██████████| 1/1 [00:00<00:00, 35.81it/s]
2024/11/05 16:40:22 INFO mlflow.tracking._tracking_service.client: 🏃 View run big-shrimp-529 at: http://localhost:5000/#/experiments/1/runs/73108c283ad44fa1be30f7f81104fefe.
2024/11/05 16:40:22 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/1.
[I 2024-11-05 16:40:22,614] Trial 41 finished with value: 8127.320001730165 and parameters: {'learning_rate': 0.003043581351302781, 'discount_factor': 0.95, 'batch_size': 128, 'memory_size': 50000, 'freq_steps_train': 8, 'freq_steps_update_target': 1000, 'n_steps_warm_up_memory': 5000, 'n_gradient_steps': 16, 'nn_hidden_layers': [256, 256], 'max_grad_norm': 10, 'normalize_state': True, 'epsilon_start': 0.9, 'epsilon_end': 0.0015151767447116624, 'steps_epsilon_decay': 10000, 'seed': 176699284}. Best is trial 35 with value: 12519.74964230619.


GPU is running
75,017 parameters


100%|██████████| 1/1 [00:00<00:00, 37.42it/s]
2024/11/05 16:54:52 INFO mlflow.tracking._tracking_service.client: 🏃 View run selective-kit-320 at: http://localhost:5000/#/experiments/1/runs/7ff1766401fc421291199ce4b51cd2e6.
2024/11/05 16:54:52 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/1.
[I 2024-11-05 16:54:52,682] Trial 42 finished with value: 7587.320001730167 and parameters: {'learning_rate': 0.0022100327455836627, 'discount_factor': 0.95, 'batch_size': 128, 'memory_size': 50000, 'freq_steps_train': 8, 'freq_steps_update_target': 1000, 'n_steps_warm_up_memory': 5000, 'n_gradient_steps': 16, 'nn_hidden_layers': [256, 256], 'max_grad_norm': 10, 'normalize_state': False, 'epsilon_start': 0.9, 'epsilon_end': 0.00041778417172537197, 'steps_epsilon_decay': 10000, 'seed': 242523794}. Best is trial 35 with value: 12519.74964230619.


GPU is running
75,017 parameters


...

GPU is running
75,017 parameters


100%|██████████| 1/1 [00:00<00:00, 35.81it/s]
2024/11/06 17:28:47 INFO mlflow.tracking._tracking_service.client: 🏃 View run righteous-donkey-194 at: http://localhost:5000/#/experiments/1/runs/d2d0fbf6c77b4b2892a8cd73bc6ee20c.
2024/11/06 17:28:47 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/1.
[I 2024-11-06 17:28:47,251] Trial 170 finished with value: 8954.263709885905 and parameters: {'learning_rate': 0.008163990584266713, 'discount_factor': 0.99, 'batch_size': 128, 'memory_size': 10000, 'freq_steps_train': 16, 'freq_steps_update_target': 1000, 'n_steps_warm_up_memory': 5000, 'n_gradient_steps': 4, 'nn_hidden_layers': [256, 256], 'max_grad_norm': 10, 'normalize_state': True, 'epsilon_start': 0.9, 'epsilon_end': 0.0902653736795002, 'steps_epsilon_decay': 10000, 'seed': 561735206}. Best is trial 35 with value: 12519.74964230619.


GPU is running
75,017 parameters


100%|██████████| 1/1 [00:00<00:00, 35.81it/s]
2024/11/06 17:31:30 INFO mlflow.tracking._tracking_service.client: 🏃 View run luxuriant-koi-327 at: http://localhost:5000/#/experiments/1/runs/fc72f2dbe083408d83ac1a71ce967803.
2024/11/06 17:31:30 INFO mlflow.tracking._tracking_service.client: 🧪 View experiment at: http://localhost:5000/#/experiments/1.
[I 2024-11-06 17:31:30,165] Trial 171 finished with value: 9367.263432212054 and parameters: {'learning_rate': 0.009986291155178979, 'discount_factor': 0.99, 'batch_size': 128, 'memory_size': 10000, 'freq_steps_train': 16, 'freq_steps_update_target': 1000, 'n_steps_warm_up_memory': 5000, 'n_gradient_steps': 4, 'nn_hidden_layers': [256, 256], 'max_grad_norm': 10, 'normalize_state': True, 'epsilon_start': 0.9, 'epsilon_end': 0.08456412960510391, 'steps_epsilon_decay': 10000, 'seed': 107090135}. Best is trial 35 with value: 12519.74964230619.


Stopping hyperparameter search because trial.value (9367.263432212054) hit threshold (9364.0)


## These are the best hyper-parameters

In [92]:
# # These best hyper-params are according 10 days-best avergage not on 301st day so will change in future
# best_trial = study.best_trial

# hparams = {k: best_trial.params[k] for k in best_trial.params if k != 'seed'}
# #hparams['nn_hidden_layers'] = eval(hparams['nn_hidden_layers']) 
# print(hparams)

# SEED = best_trial.params['seed']
# print('Seed: ', SEED)

In [93]:
# Trial 171 is best for my case correspondint to Agent # 130
# To manual put best hyper-params and training the agent again
hparams = {'learning_rate': 0.009986291155178979, 
 'discount_factor': 0.99, 
 'batch_size': 128, 
 'memory_size': 10000, 
 'freq_steps_train': 16, 
 'freq_steps_update_target': 1000, 
 'n_steps_warm_up_memory': 5000, 
 'n_gradient_steps': 4, 
 'nn_hidden_layers': [256, 256], 
 'max_grad_norm': 10, 
 'normalize_state': True, 
 'epsilon_start': 0.9, 
 'epsilon_end': 0.08456412960510391, 
 'steps_epsilon_decay': 10000, 
 }

SEED = 107090135

## We can re-run the training to get the perfect agent

In [94]:
from src.utils import set_seed
set_seed(env, SEED)

from src.q_agent import QAgent
agent = QAgent(env, **hparams)

from src.loops import train
cumrewards_per_episode, op_cost_per_episode = train(agent, env, n_episodes=2000)

GPU is running
75,017 parameters


100%|██████████| 2000/2000 [02:36<00:00, 12.75it/s]


## or simply load the `agent_id` from the best run

In [74]:
from src.q_agent import QAgent
from src.config import SAVED_AGENTS_DIR

# you can find the agent_id for the best run in the MLflow
# dashboard.
# 298 is the value in my case, but you need to check what is your
agent_id = 130

path_to_saved_model = SAVED_AGENTS_DIR / 'MgProject1' / str(agent_id)
agent = QAgent.load_from_disk(env, path_to_saved_model)

GPU is running
75,017 parameters


## Evaluate the agent ⏱️

In [5]:
DAY0 = 301
DAYN = 302

In [96]:
from src.loops import evaluate

# evaluate its performance
rewards_per_day, operation_cost_per_day = evaluate(
    agent, env, day0 = DAY0, dayn = DAYN, 
    epsilon=0.00
    )

100%|██████████| 1/1 [00:00<00:00, 55.83it/s]

Initial SOC[0]: 0.5292881023563949
Bat Ch/Dch: [-80.0, 80.0, 60.00000000000001, 48.28475905744205, 0.0, 0.0, 0.0, -80.00000000000003, -79.99999999999999, -80.0, 0.0, 80.0, 79.99999999999999, 80.00000000000003, 0.0, 0.0, -80.00000000000003, -79.99999999999999, -80.0, 0.0, 0.0, 0.0, 0.0, 0.0]
Bat SOC: [0.3292881023563949, 0.5292881023563949, 0.6792881023563949, 0.8, 0.8, 0.8, 0.8, 0.6, 0.4, 0.2, 0.2, 0.4, 0.6, 0.8, 0.8, 0.8, 0.6, 0.4, 0.2, 0.2, 0.2, 0.2, 0.2, 0.2]
Total cost: -9367.263432212054
Total cost without degradation cost: -9368.026746391626


In [77]:
import numpy as np
print(f'Reward on {DAY0} day {rewards_per_day:.2f}')

Reward on 301 day 9367.26


## Let's see how far we got in each attempt

In [51]:
# import matplotlib.pyplot as plt
# import pandas as pd

# fig, ax = plt.subplots(figsize = (10, 4))
# ax.set_title("Rewards")    
# pd.Series(rewards).plot(kind='hist', bins=100)

# plt.show()

## Let's see our agent in action 🎬

In [52]:
# Workaround for pygame error: "error: No available video device"
# See https://stackoverflow.com/questions/15933493/pygame-error-no-available-video-device?rq=1
# This is probably needed only for Linux
# import os
# os.environ["SDL_VIDEODRIVER"] = "dummy"

# from src.viz import show_video

# show_video(agent, env, sleep_sec=0.01, seed=123)

- Random agent

In [4]:
from src.random_agent import RandomAgent
agent = RandomAgent(env)

In [6]:
from src.loops import evaluate

# evaluate its performance
rewards_per_day, operation_cost_per_day = evaluate(
    agent, env, day0 = DAY0, dayn = DAYN, 
    epsilon=0.00
    )

100%|██████████| 1/1 [00:00<00:00, 111.20it/s]

Initial SOC[0]: 0.5292881023563949
Bat Ch/Dch: [60.00000000000001, 19.99999999999997, 0.0, -39.99999999999999, -39.99999999999999, -80.0, -51.71524094255795, 0.0, 0.0, 59.999999999999986, 20.000000000000018, 79.99999999999999, 0.0, 20.000000000000018, 60.00000000000001, 0.0, 0.0, 0.0, 0.0, 0.0, -40.000000000000036, 40.000000000000036, 0.0, -80.00000000000003]
Bat SOC: [0.6792881023563949, 0.7292881023563949, 0.7292881023563949, 0.6292881023563949, 0.5292881023563949, 0.3292881023563949, 0.2, 0.2, 0.2, 0.35, 0.4, 0.6, 0.6, 0.65, 0.8, 0.8, 0.8, 0.8, 0.8, 0.8, 0.7, 0.8, 0.8, 0.6]
Total cost: 810.3693116719296
Total cost without degradation cost: 809.9665670104723
